# THPT Tab 1 và Tab 5 Insights

Notebook này tạo lại các insight chính cho hai phần của dashboard:

- Tab 1: Tổng quan kỳ thi THPT Quốc gia.
- Tab 5: Tương quan môn học và khả năng phân hóa đề thi.

Phiên bản này dùng đúng pipeline dữ liệu của dashboard tại `THPT_Dashboard/data/processed`, bao gồm bảng `mavung.csv` để suy ra tỉnh/thành từ hai chữ số đầu của số báo danh. Các file dữ liệu 2025 không được dùng trong xu hướng chính vì chưa phải bộ dữ liệu năm hoàn chỉnh theo cùng cấu trúc.


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

SUBJECTS = ['toan', 'ngu_van', 'ngoai_ngu', 'vat_ly', 'hoa_hoc', 'sinh_hoc', 'lich_su', 'dia_ly', 'gdcd']
SUBJECT_NAMES = {
    'toan': 'Toán', 'ngu_van': 'Ngữ văn', 'ngoai_ngu': 'Ngoại ngữ',
    'vat_ly': 'Vật lý', 'hoa_hoc': 'Hóa học', 'sinh_hoc': 'Sinh học',
    'lich_su': 'Lịch sử', 'dia_ly': 'Địa lý', 'gdcd': 'GDCD'
}
PROVINCE_NAMES = {
    'Ha Noi': 'Hà Nội', 'TP. Ho Chi Minh': 'TP. Hồ Chí Minh', 'Hai Phong': 'Hải Phòng',
    'Da Nang': 'Đà Nẵng', 'Ha Giang': 'Hà Giang', 'Cao Bang': 'Cao Bằng',
    'Lai Chau': 'Lai Châu', 'Lao Cai': 'Lào Cai', 'Tuyen Quang': 'Tuyên Quang',
    'Lang Son': 'Lạng Sơn', 'Bac Kan': 'Bắc Kạn', 'Thai Nguyen': 'Thái Nguyên',
    'Yen Bai': 'Yên Bái', 'Son La': 'Sơn La', 'Phu Tho': 'Phú Thọ',
    'Vinh Phuc': 'Vĩnh Phúc', 'Quang Ninh': 'Quảng Ninh', 'Bac Giang': 'Bắc Giang',
    'Bac Ninh': 'Bắc Ninh', 'Hai Duong': 'Hải Dương', 'Hung Yen': 'Hưng Yên',
    'Hoa Binh': 'Hòa Bình', 'Ha Nam': 'Hà Nam', 'Nam Dinh': 'Nam Định',
    'Thai Binh': 'Thái Bình', 'Ninh Binh': 'Ninh Bình', 'Thanh Hoa': 'Thanh Hóa',
    'Nghe An': 'Nghệ An', 'Ha Tinh': 'Hà Tĩnh', 'Quang Binh': 'Quảng Bình',
    'Quang Tri': 'Quảng Trị', 'Thua Thien - Hue': 'Thừa Thiên - Huế',
    'Quang Nam': 'Quảng Nam', 'Quang Ngai': 'Quảng Ngãi', 'Kon Tum': 'Kon Tum',
    'Kon Tom': 'Kon Tum', 'Binh Dinh': 'Bình Định', 'Gia Lai': 'Gia Lai',
    'Phu Yen': 'Phú Yên', 'Dak Lak': 'Đắk Lắk', 'Khanh Hoa': 'Khánh Hòa',
    'Lam Dong': 'Lâm Đồng', 'Binh Phuoc': 'Bình Phước', 'Binh Duong': 'Bình Dương',
    'Ninh Thuan': 'Ninh Thuận', 'Tay Ninh': 'Tây Ninh', 'Binh Thuan': 'Bình Thuận',
    'Dong Nai': 'Đồng Nai', 'Long An': 'Long An', 'Dong Thap': 'Đồng Tháp',
    'An Giang': 'An Giang', 'Ba Ria - Vung Tau': 'Bà Rịa - Vũng Tàu',
    'Tien Giang': 'Tiền Giang', 'Kien Giang': 'Kiên Giang', 'Can Tho': 'Cần Thơ',
    'Ben Tre': 'Bến Tre', 'Vinh Long': 'Vĩnh Long', 'Tra Vinh': 'Trà Vinh',
    'Soc Trang': 'Sóc Trăng', 'Bac Lieu': 'Bạc Liêu', 'Ca Mau': 'Cà Mau',
    'Dien Bien': 'Điện Biên', 'Dak Nong': 'Đắk Nông', 'Hau Giang': 'Hậu Giang',
}

def resolve_data_path():
    candidates = [
        Path('THPT_Dashboard/data/processed'),
        Path('CSC10080/Final/THPT_Dashboard/data/processed'),
        Path.cwd() / 'THPT_Dashboard/data/processed',
        Path.cwd() / 'CSC10080/Final/THPT_Dashboard/data/processed',
    ]
    for path in candidates:
        if path.exists() and (path / 'mavung.csv').exists():
            return path
    raise FileNotFoundError('Cannot locate THPT_Dashboard/data/processed')

DATA_PATH = resolve_data_path()
YEARS = [2020, 2021, 2022, 2023, 2024]
DTYPE = {'sbd': str}
DTYPE.update({subject: 'float32' for subject in SUBJECTS})
frames = []

for year in YEARS:
    file_path = DATA_PATH / f'thpt{year}.csv'
    df_year = pd.read_csv(file_path, usecols=['sbd'] + SUBJECTS, dtype=DTYPE)
    df_year['year'] = year
    frames.append(df_year)
    print(f"Loaded THPT {year}: {len(df_year):,} candidates")

df_all = pd.concat(frames, ignore_index=True)

mavung = pd.read_csv(DATA_PATH / 'mavung.csv', dtype={'Ma': str})
mavung['Ma'] = mavung['Ma'].str.zfill(2)
df_all['Ma'] = df_all['sbd'].astype(str).str.zfill(8).str[:2]
df_all = df_all.merge(mavung, on='Ma', how='left')
df_all['Ten Tinh VN'] = df_all['Ten Tinh'].map(PROVINCE_NAMES).fillna(df_all['Ten Tinh'])

natural_count = df_all[['vat_ly', 'hoa_hoc', 'sinh_hoc']].notna().sum(axis=1)
social_count = df_all[['lich_su', 'dia_ly', 'gdcd']].notna().sum(axis=1)
df_all['exam_group'] = 'Thi không đầy đủ tổ hợp'
df_all.loc[(natural_count == 3) & (social_count == 0), 'exam_group'] = 'KHTN'
df_all.loc[(social_count == 3) & (natural_count == 0), 'exam_group'] = 'KHXH'

print(f"\nTotal records: {len(df_all):,}")
print(f"Data source: {DATA_PATH}")
print(f"Years used: {min(YEARS)}-{max(YEARS)}")
print(f"Subjects: {len(SUBJECTS)}")
print(f"Province mapping missing: {df_all['Ten Tinh'].isna().sum():,} records")


Loaded THPT 2020: 870,486 candidates
Loaded THPT 2021: 914,558 candidates
Loaded THPT 2022: 995,435 candidates
Loaded THPT 2023: 1,017,584 candidates
Loaded THPT 2024: 1,061,604 candidates

Total records: 4,859,667
Data source: CSC10080\Final\THPT_Dashboard\data\processed
Years used: 2020-2024
Subjects: 9
Province mapping missing: 56,738 records


## Tab 1: Tổng quan kỳ thi

### Insight 1.1: Quy mô thí sinh thay đổi như thế nào qua các năm?


In [2]:
candidates_by_year = df_all.groupby('year').size().reset_index(name='num_candidates')
candidates_by_year['yoy_change'] = candidates_by_year['num_candidates'].diff()
candidates_by_year['pct_change'] = candidates_by_year['num_candidates'].pct_change() * 100

print('INSIGHT 1.1: Candidate scale over years')
print(candidates_by_year.to_string(index=False, formatters={'pct_change': lambda x: '' if pd.isna(x) else f'{x:.2f}'}))

growth_2020_2024 = (candidates_by_year.iloc[-1]['num_candidates'] - candidates_by_year.iloc[0]['num_candidates']) / candidates_by_year.iloc[0]['num_candidates'] * 100
growth_2021_2024 = (
    candidates_by_year[candidates_by_year['year'] == 2024]['num_candidates'].iloc[0]
    - candidates_by_year[candidates_by_year['year'] == 2021]['num_candidates'].iloc[0]
) / candidates_by_year[candidates_by_year['year'] == 2021]['num_candidates'].iloc[0] * 100
avg_yoy = candidates_by_year['pct_change'].dropna().mean()

print(f"\nOverall growth, 2020-2024: {growth_2020_2024:.2f}%")
print(f"Overall growth, 2021-2024: {growth_2021_2024:.2f}%")
print(f"Average annual growth, 2020-2024: {avg_yoy:.2f}%")
print(f"Highest candidate count: {candidates_by_year.loc[candidates_by_year['num_candidates'].idxmax(), 'year']}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(candidates_by_year['year'], candidates_by_year['num_candidates'], marker='o', linewidth=2.5)
ax.fill_between(candidates_by_year['year'], candidates_by_year['num_candidates'], alpha=0.18)
ax.set_title('Candidate Scale Trend, 2020-2024')
ax.set_xlabel('Year')
ax.set_ylabel('Number of candidates')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


INSIGHT 1.1: Candidate scale over years
 year  num_candidates  yoy_change pct_change
 2020          870486         NaN        NaN
 2021          914558     44072.0       5.06
 2022          995435     80877.0       8.84
 2023         1017584     22149.0       2.23
 2024         1061604     44020.0       4.33

Overall growth, 2020-2024: 21.96%
Overall growth, 2021-2024: 16.08%
Average annual growth, 2020-2024: 5.11%
Highest candidate count: 2024


### Insight 1.2: Mức độ phân hóa điểm giữa các môn học như thế nào?


In [3]:
subject_stats = []
for subject in SUBJECTS:
    scores = df_all[subject].dropna()
    subject_stats.append({
        'Subject': SUBJECT_NAMES[subject],
        'Mean': scores.mean(),
        'Std': scores.std(),
        'Min': scores.min(),
        'Max': scores.max(),
        'Q1': scores.quantile(0.25),
        'Median': scores.quantile(0.5),
        'Q3': scores.quantile(0.75),
        'Valid': len(scores),
    })

df_stats = pd.DataFrame(subject_stats).sort_values('Mean', ascending=False)
print('INSIGHT 1.2: Subject score statistics')
print(df_stats.to_string(index=False, formatters={col: '{:.2f}'.format for col in ['Mean', 'Std', 'Min', 'Max', 'Q1', 'Median', 'Q3']}))

highest_mean = df_stats.iloc[0]
lowest_mean = df_stats.iloc[-1]
most_variable = df_stats.loc[df_stats['Std'].idxmax()]
least_variable = df_stats.loc[df_stats['Std'].idxmin()]

print(f"\nHighest average score: {highest_mean['Subject']}, {highest_mean['Mean']:.2f}")
print(f"Lowest average score: {lowest_mean['Subject']}, {lowest_mean['Mean']:.2f}")
print(f"Best score spread: {most_variable['Subject']}, std {most_variable['Std']:.2f}")
print(f"Weakest score spread: {least_variable['Subject']}, std {least_variable['Std']:.2f}")
print(f"Average score range: {highest_mean['Mean'] - lowest_mean['Mean']:.2f} points")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_stats = df_stats.sort_values('Mean')
axes[0].barh(plot_stats['Subject'], plot_stats['Mean'])
axes[0].set_title('Average Score by Subject')
axes[0].set_xlabel('Average score')
plot_std = df_stats.sort_values('Std')
axes[1].barh(plot_std['Subject'], plot_std['Std'])
axes[1].set_title('Score Spread by Subject')
axes[1].set_xlabel('Standard deviation')
plt.tight_layout()
plt.show()


INSIGHT 1.2: Subject score statistics
  Subject Mean  Std  Min   Max   Q1 Median   Q3   Valid
     GDCD 8.19 1.10 0.00 10.00 7.50   8.25 9.00 2674578
  Ngữ văn 6.75 1.36 0.00 10.00 6.00   6.75 7.75 4798302
   Địa lý 6.75 1.26 0.00 10.00 6.00   6.75 7.50 3176187
  Hóa học 6.69 1.58 0.00 10.00 5.50   7.00 8.00 1623925
   Vật lý 6.65 1.50 0.00 10.00 5.75   7.00 7.75 1616452
     Toán 6.48 1.68 0.00 10.00 5.40   6.80 7.80 4803559
  Lịch sử 5.87 1.71 0.00 10.00 4.50   6.00 7.25 3200121
 Sinh học 5.77 1.44 0.00 10.00 4.75   5.75 6.75 1600865
Ngoại ngữ 5.33 2.01 0.00 10.00 3.80   5.00 6.80 4240694

Highest average score: GDCD, 8.19
Lowest average score: Ngoại ngữ, 5.33
Best score spread: Ngoại ngữ, std 2.01
Weakest score spread: GDCD, std 1.10
Average score range: 2.87 points


### Insight 1.3: Thí sinh phân bố theo địa phương như thế nào?


In [4]:
latest_year = df_all['year'].max()
df_latest = df_all[df_all['year'] == latest_year]

top_provinces = df_latest['Ten Tinh VN'].value_counts().head(10)
bottom_provinces = df_latest['Ten Tinh VN'].value_counts().tail(5)

print(f'INSIGHT 1.3: Geographic distribution, {latest_year}')
print('\nTop provinces/cities by candidate count:')
print(top_provinces.to_string())
print('\nLowest mapped provinces by candidate count:')
print(bottom_provinces.to_string())

missing_by_year = df_all[df_all['Ten Tinh'].isna()].groupby(['year', 'Ma']).size()
print('\nUnmapped province codes:')
print(missing_by_year.to_string() if not missing_by_year.empty else 'None')


INSIGHT 1.3: Geographic distribution, 2024

Top provinces/cities by candidate count:
Ten Tinh VN
Hà Nội             107867
TP. Hồ Chí Minh     87322
Thanh Hóa           38532
Nghệ An             36729
Đồng Nai            33800
Hải Phòng           25529
Hải Dương           23366
Thái Bình           22580
Nam Định            21760
Bắc Giang           21755

Lowest mapped provinces by candidate count:
Ten Tinh VN
Ninh Thuận    6288
Cao Bằng      5506
Kon Tum       5038
Lai Châu      4188
Bắc Kạn       3174

Unmapped province codes:
year  Ma
2020  00    56738


### Insight 1.4: Xu hướng lựa chọn tổ hợp thi của học sinh ra sao?


In [5]:
group_count = pd.crosstab(df_all['year'], df_all['exam_group'])
group_pct = pd.crosstab(df_all['year'], df_all['exam_group'], normalize='index') * 100

print('INSIGHT 1.4: Exam group selection trend')
print('\nCandidate count by year:')
print(group_count.to_string())
print('\nPercentage by year:')
print(group_pct.round(2).to_string())

latest = group_pct.loc[df_all['year'].max()]
print(f"\nLatest complete year: {df_all['year'].max()}")
print(f"KHTN: {latest.get('KHTN', 0):.2f}%")
print(f"KHXH: {latest.get('KHXH', 0):.2f}%")
print(f"Thi không đầy đủ tổ hợp: {latest.get('Thi không đầy đủ tổ hợp', 0):.2f}%")


INSIGHT 1.4: Exam group selection trend

Candidate count by year:
exam_group    KHTN    KHXH  Thi không đầy đủ tổ hợp
year                                               
2020        286170  482601                   101715
2021        318205  487980                   108373
2022        318685  554090                   122660
2023        322540  565243                   129801
2024        339787  583106                   138711

Percentage by year:
exam_group   KHTN   KHXH  Thi không đầy đủ tổ hợp
year                                             
2020        32.87  55.44                    11.68
2021        34.79  53.36                    11.85
2022        32.01  55.66                    12.32
2023        31.70  55.55                    12.76
2024        32.01  54.93                    13.07

Latest complete year: 2024
KHTN: 32.01%
KHXH: 54.93%
Thi không đầy đủ tổ hợp: 13.07%


## Tab 5: Tương quan và phân hóa đề thi

### Insight 5.1: Mối quan hệ giữa các môn học trong kỳ thi như thế nào?


In [6]:
correlation_matrix = df_all[SUBJECTS].corr()
print('INSIGHT 5.1: Subject correlation matrix')
print(correlation_matrix.round(3).to_string())

print('\nStrong correlations above 0.50:')
for i, s1 in enumerate(SUBJECTS):
    for s2 in SUBJECTS[i + 1:]:
        value = correlation_matrix.loc[s1, s2]
        if pd.notna(value) and value > 0.50:
            print(f"{SUBJECT_NAMES[s1]} - {SUBJECT_NAMES[s2]}: {value:.3f}")

khtn_corr = correlation_matrix.loc[['vat_ly', 'hoa_hoc', 'sinh_hoc'], ['vat_ly', 'hoa_hoc', 'sinh_hoc']]
khxh_corr = correlation_matrix.loc[['lich_su', 'dia_ly', 'gdcd'], ['lich_su', 'dia_ly', 'gdcd']]
khtn_internal = khtn_corr.values[~np.eye(3, dtype=bool)].mean()
khxh_internal = khxh_corr.values[~np.eye(3, dtype=bool)].mean()
print(f"\nAverage internal correlation, KHTN: {khtn_internal:.3f}")
print(f"Average internal correlation, KHXH: {khxh_internal:.3f}")

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(correlation_matrix.fillna(0), vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(SUBJECTS)), [SUBJECT_NAMES[s] for s in SUBJECTS], rotation=45, ha='right')
ax.set_yticks(range(len(SUBJECTS)), [SUBJECT_NAMES[s] for s in SUBJECTS])
for row in range(len(SUBJECTS)):
    for col in range(len(SUBJECTS)):
        val = correlation_matrix.iloc[row, col]
        text = '' if pd.isna(val) else f'{val:.2f}'
        ax.text(col, row, text, ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Subject Correlation Matrix')
plt.tight_layout()
plt.show()


INSIGHT 5.1: Subject correlation matrix
            toan  ngu_van  ngoai_ngu  vat_ly  hoa_hoc  sinh_hoc  lich_su  dia_ly   gdcd
toan       1.000    0.439      0.559   0.566    0.467     0.233    0.415   0.472  0.468
ngu_van    0.439    1.000      0.370   0.161    0.176     0.293    0.456   0.445  0.439
ngoai_ngu  0.559    0.370      1.000   0.294    0.067     0.252    0.349   0.314  0.357
vat_ly     0.566    0.161      0.294   1.000    0.383     0.122      NaN     NaN    NaN
hoa_hoc    0.467    0.176      0.067   0.383    1.000     0.387      NaN     NaN    NaN
sinh_hoc   0.233    0.293      0.252   0.122    0.387     1.000      NaN     NaN    NaN
lich_su    0.415    0.456      0.349     NaN      NaN       NaN    1.000   0.559  0.482
dia_ly     0.472    0.445      0.314     NaN      NaN       NaN    0.559   1.000  0.549
gdcd       0.468    0.439      0.357     NaN      NaN       NaN    0.482   0.549  1.000

Strong correlations above 0.50:
Toán - Ngoại ngữ: 0.559
Toán - Vật lý: 0.566
Lị

### Insight 5.2: Hà Nội và TP. Hồ Chí Minh khác nhau như thế nào theo từng môn?


In [7]:
latest_year = df_all['year'].max()
df_hn_hcm = df_all[(df_all['year'] == latest_year) & (df_all['Ten Tinh VN'].isin(['Hà Nội', 'TP. Hồ Chí Minh']))]
city_counts = df_hn_hcm.groupby('Ten Tinh VN').size()

comparison = []
for subject in SUBJECTS:
    means = df_hn_hcm.groupby('Ten Tinh VN')[subject].mean()
    counts = df_hn_hcm.groupby('Ten Tinh VN')[subject].count()
    comparison.append({
        'Subject': SUBJECT_NAMES[subject],
        'Ha Noi': means.get('Hà Nội', np.nan),
        'TP HCM': means.get('TP. Hồ Chí Minh', np.nan),
        'Gap': means.get('Hà Nội', np.nan) - means.get('TP. Hồ Chí Minh', np.nan),
        'n_HN': int(counts.get('Hà Nội', 0)),
        'n_HCM': int(counts.get('TP. Hồ Chí Minh', 0)),
    })

hn_hcm_comparison = pd.DataFrame(comparison)
print(f'INSIGHT 5.2: Hanoi and Ho Chi Minh City comparison, {latest_year}')
print('\nCandidate count:')
print(city_counts.to_string())
print('\nAverage score comparison by subject:')
print(hn_hcm_comparison.to_string(index=False, formatters={'Ha Noi': '{:.2f}'.format, 'TP HCM': '{:.2f}'.format, 'Gap': '{:.2f}'.format}))
print('\nPositive gap means Ha Noi is higher. Negative gap means TP. Ho Chi Minh is higher.')


INSIGHT 5.2: Hanoi and Ho Chi Minh City comparison, 2024

Candidate count:
Ten Tinh VN
Hà Nội             107867
TP. Hồ Chí Minh     87322

Average score comparison by subject:
  Subject Ha Noi TP HCM   Gap   n_HN  n_HCM
     Toán   6.75   6.98 -0.24 106554  86491
  Ngữ văn   7.76   6.65  1.10 106636  85678
Ngoại ngữ   6.20   6.73 -0.53  89232  73288
   Vật lý   6.81   6.34  0.47  29394  49185
  Hóa học   6.22   6.49 -0.27  29349  49457
 Sinh học   5.91   6.22 -0.31  29006  49065
  Lịch sử   6.62   6.62 -0.00  76953  36799
   Địa lý   7.06   7.20 -0.14  76875  36613
     GDCD   8.12   8.31 -0.19  62765  29330

Positive gap means Ha Noi is higher. Negative gap means TP. Ho Chi Minh is higher.


### Insight 5.3: Đề thi có phân hóa học sinh tốt hay không?


In [8]:
df_disc = pd.DataFrame(subject_stats)
mean_score = df_disc['Mean'].mean()
mean_std = df_disc['Std'].mean()

def classify_exam_quality(row):
    difficulty = 'Dễ' if row['Mean'] >= mean_score else 'Khó'
    discrimination = 'phân hóa tốt' if row['Std'] >= mean_std else 'phân hóa kém'
    return f'{difficulty} và {discrimination}'

df_disc['Quality'] = df_disc.apply(classify_exam_quality, axis=1)
df_disc = df_disc.sort_values('Mean', ascending=False)

print('INSIGHT 5.3: Exam discrimination quality')
print(f"Thresholds used: average subject mean = {mean_score:.2f}, average subject std = {mean_std:.2f}")
print('High average score is interpreted as an easier paper; low average score is interpreted as a harder paper.\n')
print(df_disc[['Subject', 'Mean', 'Std', 'Valid', 'Quality']].to_string(index=False, formatters={'Mean': '{:.2f}'.format, 'Std': '{:.2f}'.format}))
print('\nQuality counts:')
print(df_disc['Quality'].value_counts().to_string())
print(f"\nSubjects with good discrimination: {df_disc['Quality'].str.contains('phân hóa tốt').sum()}/{len(df_disc)}")


INSIGHT 5.3: Exam discrimination quality
Thresholds used: average subject mean = 6.50, average subject std = 1.52
High average score is interpreted as an easier paper; low average score is interpreted as a harder paper.

  Subject Mean  Std   Valid             Quality
     GDCD 8.19 1.10 2674578  Dễ và phân hóa kém
  Ngữ văn 6.75 1.36 4798302  Dễ và phân hóa kém
   Địa lý 6.75 1.26 3176187  Dễ và phân hóa kém
  Hóa học 6.69 1.58 1623925  Dễ và phân hóa tốt
   Vật lý 6.65 1.50 1616452  Dễ và phân hóa kém
     Toán 6.48 1.68 4803559 Khó và phân hóa tốt
  Lịch sử 5.87 1.71 3200121 Khó và phân hóa tốt
 Sinh học 5.77 1.44 1600865 Khó và phân hóa kém
Ngoại ngữ 5.33 2.01 4240694 Khó và phân hóa tốt

Quality counts:
Quality
Dễ và phân hóa kém     4
Khó và phân hóa tốt    3
Dễ và phân hóa tốt     1
Khó và phân hóa kém    1

Subjects with good discrimination: 4/9


## Executive Summary


In [9]:
total_candidates = len(df_all)
latest_year = df_all['year'].max()
latest_group_pct = group_pct.loc[latest_year]
good_disc_count = df_disc['Quality'].str.contains('phân hóa tốt').sum()

print('EXECUTIVE SUMMARY')
print('\nTab 1:')
print(f'Total candidates, {min(YEARS)}-{max(YEARS)}: {total_candidates:,}')
print(f'Candidate scale increased by {growth_2020_2024:.2f}% from {min(YEARS)} to {max(YEARS)}.')
print(f"Highest average score: {highest_mean['Subject']} ({highest_mean['Mean']:.2f}).")
print(f"Lowest average score and strongest score spread: {lowest_mean['Subject']} ({lowest_mean['Mean']:.2f}, std {most_variable['Std']:.2f}).")
print('Ha Noi and TP. Ho Chi Minh are the two largest candidate centers in 2024.')
print(f"KHXH remains dominant in {latest_year}: {latest_group_pct.get('KHXH', 0):.2f}% versus {latest_group_pct.get('KHTN', 0):.2f}% for KHTN.")

print('\nTab 5:')
print('The strongest relationships are Toán - Vật lý, Toán - Ngoại ngữ, Lịch sử - Địa lý, and Địa lý - GDCD.')
print(f'KHXH subjects are more internally connected than KHTN subjects ({khxh_internal:.3f} vs {khtn_internal:.3f}).')
print('In 2024, Ha Noi is clearly higher in Ngữ văn and Vật lý; TP. Ho Chi Minh is higher in most remaining subjects.')
print(f'Subjects with good discrimination: {good_disc_count}/{len(df_disc)}.')


EXECUTIVE SUMMARY

Tab 1:
Total candidates, 2020-2024: 4,859,667
Candidate scale increased by 21.96% from 2020 to 2024.
Highest average score: GDCD (8.19).
Lowest average score and strongest score spread: Ngoại ngữ (5.33, std 2.01).
Ha Noi and TP. Ho Chi Minh are the two largest candidate centers in 2024.
KHXH remains dominant in 2024: 54.93% versus 32.01% for KHTN.

Tab 5:
The strongest relationships are Toán - Vật lý, Toán - Ngoại ngữ, Lịch sử - Địa lý, and Địa lý - GDCD.
KHXH subjects are more internally connected than KHTN subjects (0.530 vs 0.297).
In 2024, Ha Noi is clearly higher in Ngữ văn and Vật lý; TP. Ho Chi Minh is higher in most remaining subjects.
Subjects with good discrimination: 4/9.
